# Model Optimization: Knowledge Distillation

In this notebook, we'll apply knowledge distillation to our models using distributed processing. Knowledge distillation is a technique where a smaller "student" model is trained to mimic the behavior of a larger "teacher" model.

## What is Knowledge Distillation?

Knowledge distillation is a model compression technique where a small model (student) is trained to mimic a larger, more complex model (teacher). The key insight is that the teacher model's outputs contain rich information beyond just the predicted class - they contain the relative probabilities across all classes, which represent the teacher's "dark knowledge".

### How Knowledge Distillation Works:

1. **Teacher Model**: A large, pre-trained model with high accuracy but high computational requirements
2. **Student Model**: A smaller model architecture that we want to train
3. **Distillation Process**: The student is trained using a combination of:
   - **Hard Targets**: The actual ground truth labels (standard supervised learning)
   - **Soft Targets**: The probability distributions output by the teacher model

### Benefits of Knowledge Distillation:
- **Reduced Model Size**: Student models are typically much smaller than teacher models
- **Faster Inference**: Smaller models require less computation for predictions
- **Lower Memory Requirements**: Smaller models use less memory during inference
- **Preserved Accuracy**: Student models often retain much of the teacher's performance

### Distributed Processing Approach
This notebook uses SageMaker Processing jobs to perform knowledge distillation on separate, more powerful instances. This approach allows us to:
1. Use a small, cost-effective instance for our notebook
2. Launch larger instances only when needed for resource-intensive tasks
3. Process multiple models in parallel

## 1. Import Dependencies

In [ ]:
# Import required libraries
import os
import sys
import time
import json
import boto3
import sagemaker
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Import specific modules for this notebook
from sagemaker.processing import ProcessingInput, ProcessingOutput
from sagemaker.pytorch.processing import PyTorchProcessor
from IPython.display import clear_output

## 2. Load Workshop Settings

In [ ]:
# Load stored variables
%store -r S3_BUCKET
%store -r AWS_REGION
%store -r SAGEMAKER_ROLE_ARN
%store -r OPTIMIZATION_INSTANCE_TYPE

# Check if variables were successfully retrieved
if 'S3_BUCKET' in locals() and S3_BUCKET != "YOUR_BUCKET_NAME_HERE":
    print("Workshop settings loaded successfully:")
    print(f"S3 Bucket: {S3_BUCKET}")
    print(f"AWS Region: {AWS_REGION}")
    print(f"SageMaker Role ARN: {SAGEMAKER_ROLE_ARN}")
    print(f"Optimization Instance Type: {OPTIMIZATION_INSTANCE_TYPE}")
else:
    print("⚠️ Workshop settings not found or not configured.")
    print("Please run the first notebook (01_introduction_and_setup.ipynb) to configure settings.")

# Initialize S3 client
s3_client = boto3.client('s3')

## 3. Load Model Information

In [ ]:
# Load model information from previous notebooks
try:
    with open('model_info.json', 'r') as f:
        model_info_dict = json.load(f)
    print(f"Loaded model information for {len(model_info_dict)} models")
except FileNotFoundError:
    print("model_info.json not found. Creating default model info.")
    model_info_dict = {
        "sentiment-analysis": {
            "model_name": "distilbert-base-uncased-finetuned-sst-2-english",
            "task": "text-classification",
            "hub_model_id": "distilbert-base-uncased-finetuned-sst-2-english",
            "s3_uri": f"s3://{S3_BUCKET}/models/distilbert-base-uncased-finetuned-sst-2-english"
        }
    }
    
    # Save model info to file
    with open('model_info.json', 'w') as f:
        json.dump(model_info_dict, f, indent=2)
    print("Created default model info with sentiment analysis model")

# Display model info
for model_key, info in model_info_dict.items():
    print(f"\nModel: {model_key}")
    print(f"  Name: {info['model_name']}")
    print(f"  Task: {info['task']}")
    print(f"  S3 URI: {info.get('s3_uri', 'Not available')}")

## 4. Configure Knowledge Distillation Jobs

Knowledge distillation works by training a smaller "student" model to mimic a larger "teacher" model. We'll configure SageMaker Processing jobs to perform this distillation process.

In [ ]:
# Load workshop configuration
with open('workshop_config.json', 'r') as f:
    workshop_config = json.load(f)

# Set up SageMaker session
sagemaker_session = sagemaker.Session()
role = workshop_config['role']
region = workshop_config['region']
bucket = workshop_config['s3_bucket']
prefix = workshop_config['s3_prefix']

print(f"SageMaker session established in region: {region}")
print(f"Using S3 bucket: {bucket}")
print(f"Using S3 prefix: {prefix}")

# Create PyTorch processor for distillation
processor = PyTorchProcessor(
    framework_version='2.4.0',
    py_version='py310',
    role=role,
    instance_type=OPTIMIZATION_INSTANCE_TYPE,
    instance_count=1,
    base_job_name='knowledge-distillation',
    sagemaker_session=sagemaker_session
)

print(f"Created PyTorch processor with instance type: {OPTIMIZATION_INSTANCE_TYPE}")